In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math
import gc
import os
import sys
import functools
os.environ["HF_HOME"] = "/cs/student/project_msc/2025/csml/katayama/huggingface_cache/stage1_step100000"
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from tqdm.std import tqdm
import itertools
import glob
import re
import numpy as np
from sklearn.decomposition import IncrementalPCA
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

In [2]:
# Import custom module
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

%load_ext autoreload
%autoreload 2

from modules.experiment_helpers import set_seed

In [3]:
model_id = "allenai/OLMo-3-1025-7B"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
subject = "all"
batch_size = 2
Trained = False
Parameter_Obtained = True
set_seed(42)

cuda


In [4]:
if not Trained:
    print(f"loading {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision = "stage1-step100000")
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.bfloat16,
        device_map="auto", 
        quantization_config=quantization_config, 
        revision = "stage1-step100000"
    )
    model.eval()

loading allenai/OLMo-3-1025-7B...


/cs/student/project_msc/2025/csml/katayama/dl_env/lib64/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
if not Trained: 
    print(f"loading MMLU ({subject})...")
    dataset = load_dataset("cais/mmlu", subject, split="test")

loading MMLU (all)...


In [6]:
if not Trained: 
    choices = ["A", "B", "C", "D"]
    choice_ids = [tokenizer.encode(c, add_special_tokens=False)[-1] for c in choices]
    prompt_template = "Question: {question}\nA. {a}\nB. {b}\nC. {c}\nD. {d}\nAnswer: "

In [7]:
if not Trained:
    prompts = []
    answers = []
    for sample in dataset:
        p = prompt_template.format(
            question=sample['question'],
            a=sample['choices'][0],
            b=sample['choices'][1],
            c=sample['choices'][2],
            d=sample['choices'][3]
        )
        prompts.append(p)
        answers.append(sample['answer'])

    correct_count = 0
    total_count = len(dataset)

In [8]:
if not Trained:
    # Left padding for batch processing
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Get token IDs for the choices
    choices = ["A", "B", "C", "D"]
    choice_ids = [tokenizer.encode(c, add_special_tokens=False)[-1] for c in choices]

    correct_count = 0
    total_count = len(dataset)

    print("="*50)
    print(f"Starting MMLU Accuracy Evaluation (Logit-based)...")
    print(f"Total questions: {total_count}")
    print("="*50)

    for i in range(0, total_count, batch_size):
        batch_prompts = prompts[i:i+batch_size]
        batch_answers = answers[i:i+batch_size]
        
        if not batch_prompts: continue

        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        try:
            # --------------------------------------------------
            # STEP 1: Get logits via forward pass
            # --------------------------------------------------
            with torch.inference_mode():
                outputs = model(**inputs)
                
            # --------------------------------------------------
            # STEP 2: Accuracy evaluation based on logits
            # --------------------------------------------------
            next_token_logits = outputs.logits[:, -1, :]
            
            # Extract the logits corresponding to A, B, C, and D from the entire vocabulary
            choice_logits = next_token_logits[:, choice_ids]
            
            # Get the index (0, 1, 2, 3) of the highest logit among the four choices
            predictions = torch.argmax(choice_logits, dim=-1)
            
            # Compare with the ground truth answers
            for j in range(len(batch_prompts)):
                if predictions[j].item() == batch_answers[j]:
                    correct_count += 1
                    
        except torch.cuda.OutOfMemoryError:
            print(f"\nOOM Error at batch {i}. Skipping...")
            pass
        finally:
            torch.cuda.empty_cache()
            gc.collect()

    # ==========================================
    # Final Accuracy Report
    # ==========================================
    if total_count > 0:
        accuracy = correct_count / total_count
        print("\n" + "="*50)
        print(f"Subject: {subject}")
        print(f"Total Questions: {total_count}")
        print(f"Correct Answers: {correct_count}")
        print(f"Accuracy: {accuracy * 100:.2f}%")
        print("="*50)

Starting MMLU Accuracy Evaluation (Logit-based)...
Total questions: 14042



Subject: all
Total Questions: 14042
Correct Answers: 5694
Accuracy: 40.55%
